In [2]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np



In [3]:
PROJECT_ROOT = Path.cwd().parent
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"


con = duckdb.connect()

In [4]:
jan_path = INTERIM_DIR / "flights_2025_01.parquet"
print(jan_path.exists())
print(jan_path)


True
/Users/tseringgurung/Desktop/flight-operations-intelligence/data/interim/flights_2025_01.parquet


## Temporal Modeling Strategy

The model will be evaluated using chronological rather than random data splits.

Planned initial structure:

- Training: January–September 2025
- Validation: October–November 2025
- Test: December 2025

This prevents future observations from leaking into model training and better represents real operational deployment.

In [5]:
RAW_FLIGHT_DIR = PROJECT_ROOT / "data" / "raw" / "flights"

FEB_RAW_PATH = RAW_FLIGHT_DIR / "bts_ontime_2025_02.csv"

print(FEB_RAW_PATH.exists())
print(FEB_RAW_PATH)

True
/Users/tseringgurung/Desktop/flight-operations-intelligence/data/raw/flights/bts_ontime_2025_02.csv


In [6]:
con.execute(
    f"""
    CREATE OR REPLACE VIEW flights_feb_raw AS
    SELECT *
    FROM read_csv_auto(
        '{FEB_RAW_PATH}',
        sample_size = 100000,
        ignore_errors = true
    )
    """
)

In [7]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    COUNT(DISTINCT FlightDate) AS days
FROM flights_feb_raw
""").df()

,total_rows,first_date,last_date,days
0,504884,2025-02-01,2025-02-28,28


In [8]:
jan_schema = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{jan_path}')
    """
).df()

feb_schema = con.sql("""
DESCRIBE flights_feb_raw
""").df()

print("January columns:", len(jan_schema))
print("February columns:", len(feb_schema))

January columns: 110
February columns: 110


In [9]:
jan_columns = set(jan_schema["column_name"])
feb_columns = set(feb_schema["column_name"])

print("Missing in February:")
print(jan_columns - feb_columns)

print("\nNew in February:")
print(feb_columns - jan_columns)

Missing in February:
set()

New in February:
set()


In [10]:
same_order = (
    jan_schema["column_name"].tolist()
    == feb_schema["column_name"].tolist()
)

same_order

True

In [11]:
FEB_PARQUET_PATH = (
    PROJECT_ROOT 
    / "Data"
    / "INTERIM"
    /"flights_2025_02.parquet"
)

In [12]:
con.execute(
    f"""
    COPY (
        SELECT *
        FROM flights_feb_raw
    )
    TO '{FEB_PARQUET_PATH}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

In [13]:
print(FEB_PARQUET_PATH.exists())

con.sql(
    f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{FEB_PARQUET_PATH}')
    """
).df()

True


,total_rows
0,504884


In [14]:
feb_csv_size_mb = FEB_RAW_PATH.stat().st_size / 1024**2
feb_parquet_size_mb = FEB_PARQUET_PATH.stat().st_size / 1024**2

print(f"CSV size:     {feb_csv_size_mb:.2f} MB")
print(f"Parquet size: {feb_parquet_size_mb:.2f} MB")
print(
    f"Reduction:    "
    f"{(1 - feb_parquet_size_mb / feb_csv_size_mb) * 100:.1f}%"
)

CSV size:     217.67 MB
Parquet size: 12.06 MB
Reduction:    94.5%


In [15]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [16]:
from src.ingest_bts import process_bts_month

In [17]:
feb_result = process_bts_month(
    csv_path=FEB_RAW_PATH,
    parquet_path=FEB_PARQUET_PATH,
    expected_year=2025,
    expected_month=2,
)

feb_result

{'year': 2025,
 'month': 2,
 'rows': 504884,
 'first_date': Timestamp('2025-02-01 00:00:00'),
 'last_date': Timestamp('2025-02-28 00:00:00'),
 'days': 28,
 'csv_size_mb': 217.67,
 'parquet_size_mb': 12.06,
 'storage_reduction_pct': 94.5}

In [18]:
MAR_RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "flights"
    / "bts_ontime_2025_03.csv"
)

MAR_PARQUET_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "flights_2025_03.parquet"
)

print(MAR_RAW_PATH.exists())

True


In [19]:
mar_result = process_bts_month(
    csv_path=MAR_RAW_PATH,
    parquet_path=MAR_PARQUET_PATH,
    expected_year=2025,
    expected_month=3,
)

mar_result

{'year': 2025,
 'month': 3,
 'rows': 600872,
 'first_date': Timestamp('2025-03-01 00:00:00'),
 'last_date': Timestamp('2025-03-31 00:00:00'),
 'days': 31,
 'csv_size_mb': 259.04,
 'parquet_size_mb': 14.58,
 'storage_reduction_pct': 94.4}

In [20]:
mar_result = process_bts_month(
    csv_path=MAR_RAW_PATH,
    parquet_path=MAR_PARQUET_PATH,
    expected_year=2025,
    expected_month=3,
    reference_parquet_path=jan_path,
)

mar_result

{'year': 2025,
 'month': 3,
 'rows': 600872,
 'first_date': Timestamp('2025-03-01 00:00:00'),
 'last_date': Timestamp('2025-03-31 00:00:00'),
 'days': 31,
 'csv_size_mb': 259.04,
 'parquet_size_mb': 14.58,
 'storage_reduction_pct': 94.4}

In [21]:
RAW_FLIGHT_DIR = PROJECT_ROOT / "data" / "raw" / "flights"

for month in range(1, 13):
    path = RAW_FLIGHT_DIR / f"bts_ontime_2025_{month:02d}.csv"
    print(f"{month:02d}: {path.exists()}")

01: True
02: True
03: True
04: True
05: True
06: True
07: True
08: True
09: True
10: True
11: True
12: True


In [22]:
ingestion_results = []

for month in range(4, 13):

    csv_path = (
        RAW_FLIGHT_DIR
        / f"bts_ontime_2025_{month:02d}.csv"
    )

    parquet_path = (
        INTERIM_DIR
        / f"flights_2025_{month:02d}.parquet"
    )

    print(f"Processing 2025-{month:02d}...")

    result = process_bts_month(
        csv_path=csv_path,
        parquet_path=parquet_path,
        expected_year=2025,
        expected_month=month,
        reference_parquet_path=jan_path,
    )

    ingestion_results.append(result)

    print(
        f"✓ {result['rows']:,} rows | "
        f"{result['storage_reduction_pct']}% reduction"
    )

Processing 2025-04...
✓ 583,950 rows | 94.5% reduction
Processing 2025-05...
✓ 605,648 rows | 94.5% reduction
Processing 2025-06...
✓ 611,575 rows | 94.5% reduction
Processing 2025-07...
✓ 631,428 rows | 94.4% reduction
Processing 2025-08...
✓ 602,378 rows | 94.4% reduction
Processing 2025-09...
✓ 562,439 rows | 94.6% reduction
Processing 2025-10...
✓ 605,844 rows | 94.5% reduction
Processing 2025-11...
✓ 570,550 rows | 94.4% reduction
Processing 2025-12...
✓ 582,304 rows | 94.1% reduction


In [23]:
ingestion_summary = pd.DataFrame(ingestion_results)

ingestion_summary

,year,month,rows,first_date,last_date,days,csv_size_mb,parquet_size_mb,storage_reduction_pct
0,2025,4,583950,2025-04-01,2025-04-30,30,251.85,13.96,94.5
1,2025,5,605648,2025-05-01,2025-05-31,31,261.62,14.33,94.5
2,2025,6,611575,2025-06-01,2025-06-30,30,264.55,14.64,94.5
3,2025,7,631428,2025-07-01,2025-07-31,31,272.92,15.21,94.4
4,2025,8,602378,2025-08-01,2025-08-31,31,260.06,14.49,94.4
5,2025,9,562439,2025-09-01,2025-09-30,30,242.30,13.14,94.6
6,2025,10,605844,2025-10-01,2025-10-31,31,261.98,14.42,94.5
7,2025,11,570550,2025-11-01,2025-11-30,30,246.12,13.84,94.4
8,2025,12,582304,2025-12-01,2025-12-31,31,252.27,14.90,94.1


In [24]:
all_flights_path = INTERIM_DIR / "flights_2025_*.parquet"

con.execute(
    f"""
    CREATE OR REPLACE VIEW flights_2025 AS
    SELECT *
    FROM read_parquet('{all_flights_path}')
    """
)

In [25]:
year_validation = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    COUNT(DISTINCT FlightDate) AS days,
    COUNT(DISTINCT Month) AS months
FROM flights_2025
""").df()

year_validation

,total_rows,first_date,last_date,days,months
0,7001619,2025-01-01,2025-12-31,365,12


In [26]:
monthly_counts = con.sql("""
SELECT
    Month,
    COUNT(*) AS flights
FROM flights_2025
GROUP BY Month
ORDER BY Month
""").df()

monthly_counts

,Month,flights
0,1,539747
1,2,504884
2,3,600872
3,4,583950
4,5,605648
5,6,611575
6,7,631428
7,8,602378
8,9,562439
9,10,605844


In [27]:
modeling_2025 = con.sql("""
SELECT
    Year,
    Quarter,
    Month,
    DayofMonth,
    DayOfWeek,
    FlightDate,

    Reporting_Airline,
    Origin,
    Dest,

    CRSDepTime,
    CRSArrTime,
    CRSElapsedTime,

    Distance,
    DistanceGroup,

    Origin || '_' || Dest AS route,

    CAST(
        FLOOR(CAST(CRSDepTime AS INTEGER) / 100)
        AS INTEGER
    ) AS scheduled_dep_hour,

    CAST(
        FLOOR(CAST(CRSArrTime AS INTEGER) / 100)
        AS INTEGER
    ) AS scheduled_arr_hour,

    CASE
        WHEN DayOfWeek IN (6, 7) THEN 1
        ELSE 0
    END AS is_weekend,

    CASE
        WHEN CAST(CRSDepTime AS INTEGER) < 600 THEN 'overnight'
        WHEN CAST(CRSDepTime AS INTEGER) < 1200 THEN 'morning'
        WHEN CAST(CRSDepTime AS INTEGER) < 1800 THEN 'afternoon'
        ELSE 'evening'
    END AS departure_period,

    ArrDelay,

    CASE
        WHEN ArrDelay >= 15 THEN 1
        ELSE 0
    END AS significant_arrival_delay

FROM flights_2025

WHERE Cancelled = 0
  AND Diverted = 0
  AND ArrDelay IS NOT NULL
""")

In [28]:
modeling_2025.aggregate("""
    COUNT(*) AS modeling_rows
""").df()

,modeling_rows
0,6879484


In [29]:
year_target = con.sql("""
SELECT
    significant_arrival_delay,
    COUNT(*) AS flights,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM modeling_2025
GROUP BY significant_arrival_delay
ORDER BY significant_arrival_delay
""").df()

year_target

,significant_arrival_delay,flights,percentage
0,0,5344846,77.69
1,1,1534638,22.31


In [30]:
monthly_target = con.sql("""
SELECT
    Month,
    COUNT(*) AS flights,
    SUM(significant_arrival_delay) AS delayed_flights,
    ROUND(
        AVG(significant_arrival_delay) * 100,
        2
    ) AS delay_rate_pct
FROM modeling_2025
GROUP BY Month
ORDER BY Month
""").df()

monthly_target

,Month,flights,delayed_flights,delay_rate_pct
0,1,522269,98130.0,18.79
1,2,496476,103102.0,20.77
2,3,592301,116034.0,19.59
3,4,577730,113604.0,19.66
4,5,597574,141038.0,23.60
5,6,599472,169405.0,28.26
6,7,612811,177012.0,28.89
7,8,593733,133920.0,22.56
8,9,558328,92859.0,16.63
9,10,601570,122251.0,20.32


In [31]:
train_2025 = con.sql("""
SELECT *
FROM modeling_2025
WHERE Month BETWEEN 1 AND 9
""")

validation_2025 = con.sql("""
SELECT *
FROM modeling_2025
WHERE Month BETWEEN 10 AND 11
""")

test_2025 = con.sql("""
SELECT *
FROM modeling_2025
WHERE Month = 12
""")

In [32]:
split_summary = con.sql("""
SELECT
    'train' AS split,
    COUNT(*) AS flights,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    ROUND(AVG(significant_arrival_delay) * 100, 2) AS delay_rate_pct
FROM train_2025

UNION ALL

SELECT
    'validation',
    COUNT(*),
    MIN(FlightDate),
    MAX(FlightDate),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM validation_2025

UNION ALL

SELECT
    'test',
    COUNT(*),
    MIN(FlightDate),
    MAX(FlightDate),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM test_2025
""").df()

split_summary

,split,flights,first_date,last_date,delay_rate_pct
0,train,5150694,2025-01-01,2025-09-30,22.23
1,validation,1156866,2025-10-01,2025-11-30,20.44
2,test,571924,2025-12-01,2025-12-31,26.77


In [34]:
carrier_daily = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY
    FlightDate,
    Reporting_Airline
""")
carrier_daily


┌────────────┬───────────────────┬───────────────┬───────────────────────┐
│ FlightDate │ Reporting_Airline │ daily_flights │ daily_delayed_flights │
│    date    │      varchar      │     int64     │        int128         │
├────────────┼───────────────────┼───────────────┼───────────────────────┤
│ 2025-01-16 │ AA                │          2596 │                   371 │
│ 2025-01-26 │ AA                │          2547 │                   400 │
│ 2025-01-07 │ AS                │           484 │                   102 │
│ 2025-01-11 │ AS                │           481 │                    87 │
│ 2025-01-19 │ AS                │           611 │                   115 │
│ 2025-01-21 │ AS                │           488 │                    63 │
│ 2025-04-06 │ MQ                │           831 │                   213 │
│ 2025-04-11 │ MQ                │           840 │                   105 │
│ 2025-04-14 │ MQ                │           840 │                    82 │
│ 2025-04-06 │ OH        

In [35]:
carrier_history = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,

    SUM(daily_flights) OVER (
        PARTITION BY Reporting_Airline
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_carrier_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY Reporting_Airline
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_carrier_delays

FROM carrier_daily
""")

In [36]:
carrier_history_rates = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,
    prior_carrier_flights,
    prior_carrier_delays,

    CASE
        WHEN prior_carrier_flights > 0
        THEN prior_carrier_delays * 1.0 / prior_carrier_flights
        ELSE NULL
    END AS carrier_historical_delay_rate

FROM carrier_history
""")

In [38]:
con.sql("""
SELECT *
FROM carrier_history_rates
WHERE Reporting_Airline = 'AA'
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,Reporting_Airline,prior_carrier_flights,prior_carrier_delays,carrier_historical_delay_rate
0,2025-01-01,AA,NaN,NaN,NaN
1,2025-01-02,AA,2403.0,299.0,0.124428
2,2025-01-03,AA,5057.0,718.0,0.141981
3,2025-01-04,AA,7631.0,1264.0,0.165640
4,2025-01-05,AA,10001.0,1831.0,0.183082
5,2025-01-06,AA,12427.0,2812.0,0.226281
6,2025-01-07,AA,14778.0,3960.0,0.267966
7,2025-01-08,AA,16969.0,4319.0,0.254523
8,2025-01-09,AA,19195.0,4681.0,0.243866
9,2025-01-10,AA,21028.0,5106.0,0.242819


In [39]:
origin_daily = con.sql("""
SELECT
    FlightDate,
    Origin,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY
    FlightDate,
    Origin
""")

In [40]:
origin_history = con.sql("""
SELECT
    FlightDate,
    Origin,

    SUM(daily_flights) OVER (
        PARTITION BY Origin
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_origin_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY Origin
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_origin_delays

FROM origin_daily
""")

In [41]:
origin_history_rates = con.sql("""
SELECT
    FlightDate,
    Origin,
    prior_origin_flights,
    prior_origin_delays,

    CASE
        WHEN prior_origin_flights > 0
        THEN prior_origin_delays * 1.0 / prior_origin_flights
        ELSE NULL
    END AS origin_historical_delay_rate

FROM origin_history
""")

In [42]:
con.sql("""
SELECT *
FROM origin_history_rates
WHERE Origin = 'ATL'
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,Origin,prior_origin_flights,prior_origin_delays,origin_historical_delay_rate
0,2025-01-01,ATL,NaN,NaN,NaN
1,2025-01-02,ATL,668.0,93.0,0.139222
2,2025-01-03,ATL,1461.0,249.0,0.170431
3,2025-01-04,ATL,2263.0,402.0,0.177640
4,2025-01-05,ATL,3023.0,647.0,0.214026
5,2025-01-06,ATL,3780.0,904.0,0.239153
6,2025-01-07,ATL,4511.0,1151.0,0.255154
7,2025-01-08,ATL,5213.0,1301.0,0.249568
8,2025-01-09,ATL,5929.0,1419.0,0.239332
9,2025-01-10,ATL,6662.0,1599.0,0.240018


In [43]:
destination_daily = con.sql("""
SELECT
    FlightDate,
    Dest,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY
    FlightDate,
    Dest
""")

In [44]:
destination_history = con.sql("""
SELECT
    FlightDate,
    Dest,

    SUM(daily_flights) OVER (
        PARTITION BY Dest
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_destination_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY Dest
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_destination_delays

FROM destination_daily
""")

In [45]:
destination_history_rates = con.sql("""
SELECT
    FlightDate,
    Dest,
    prior_destination_flights,
    prior_destination_delays,

    CASE
        WHEN prior_destination_flights > 0
        THEN prior_destination_delays * 1.0 / prior_destination_flights
        ELSE NULL
    END AS destination_historical_delay_rate

FROM destination_history
""")

In [46]:
con.sql("""
SELECT *
FROM destination_history_rates
WHERE Dest = 'ATL'
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,Dest,prior_destination_flights,prior_destination_delays,destination_historical_delay_rate
0,2025-01-01,ATL,NaN,NaN,NaN
1,2025-01-02,ATL,663.0,76.0,0.114630
2,2025-01-03,ATL,1465.0,195.0,0.133106
3,2025-01-04,ATL,2265.0,335.0,0.147903
4,2025-01-05,ATL,3031.0,526.0,0.173540
5,2025-01-06,ATL,3806.0,772.0,0.202838
6,2025-01-07,ATL,4515.0,1003.0,0.222148
7,2025-01-08,ATL,5213.0,1121.0,0.215039
8,2025-01-09,ATL,5926.0,1221.0,0.206041
9,2025-01-10,ATL,6635.0,1327.0,0.200000


In [47]:
route_daily = con.sql("""
SELECT
    FlightDate,
    route,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY
    FlightDate,
    route
""")

In [48]:
route_history = con.sql("""
SELECT
    FlightDate,
    route,

    SUM(daily_flights) OVER (
        PARTITION BY route
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_route_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY route
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_route_delays

FROM route_daily
""")

In [49]:
route_history_rates = con.sql("""
SELECT
    FlightDate,
    route,
    prior_route_flights,
    prior_route_delays,

    CASE
        WHEN prior_route_flights > 0
        THEN prior_route_delays * 1.0 / prior_route_flights
        ELSE NULL
    END AS route_historical_delay_rate

FROM route_history
""")

In [50]:
con.sql("""
SELECT *
FROM route_history_rates
WHERE route = 'ATL_MCO'
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,route,prior_route_flights,prior_route_delays,route_historical_delay_rate
0,2025-01-01,ATL_MCO,NaN,NaN,NaN
1,2025-01-02,ATL_MCO,19.0,1.0,0.052632
2,2025-01-03,ATL_MCO,42.0,4.0,0.095238
3,2025-01-04,ATL_MCO,66.0,8.0,0.121212
4,2025-01-05,ATL_MCO,89.0,18.0,0.202247
5,2025-01-06,ATL_MCO,114.0,25.0,0.219298
6,2025-01-07,ATL_MCO,135.0,31.0,0.229630
7,2025-01-08,ATL_MCO,155.0,37.0,0.238710
8,2025-01-09,ATL_MCO,175.0,44.0,0.251429
9,2025-01-10,ATL_MCO,197.0,48.0,0.243655


In [51]:
modeling_with_history = con.sql("""
SELECT
    m.*,

    c.prior_carrier_flights,
    c.carrier_historical_delay_rate,

    o.prior_origin_flights,
    o.origin_historical_delay_rate,

    d.prior_destination_flights,
    d.destination_historical_delay_rate,

    r.prior_route_flights,
    r.route_historical_delay_rate

FROM modeling_2025 m

LEFT JOIN carrier_history_rates c
    ON m.FlightDate = c.FlightDate
   AND m.Reporting_Airline = c.Reporting_Airline

LEFT JOIN origin_history_rates o
    ON m.FlightDate = o.FlightDate
   AND m.Origin = o.Origin

LEFT JOIN destination_history_rates d
    ON m.FlightDate = d.FlightDate
   AND m.Dest = d.Dest

LEFT JOIN route_history_rates r
    ON m.FlightDate = r.FlightDate
   AND m.route = r.route
""")

In [52]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM modeling_2025) AS original_rows,
    (SELECT COUNT(*) FROM modeling_with_history) AS joined_rows
""").df()

,original_rows,joined_rows
0,6879484,6879484


In [53]:
history_missingness = con.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN carrier_historical_delay_rate IS NULL THEN 1 ELSE 0 END)
        AS missing_carrier_history,

    SUM(CASE WHEN origin_historical_delay_rate IS NULL THEN 1 ELSE 0 END)
        AS missing_origin_history,

    SUM(CASE WHEN destination_historical_delay_rate IS NULL THEN 1 ELSE 0 END)
        AS missing_destination_history,

    SUM(CASE WHEN route_historical_delay_rate IS NULL THEN 1 ELSE 0 END)
        AS missing_route_history

FROM modeling_with_history
""").df()

history_missingness

,total_rows,missing_carrier_history,missing_origin_history,missing_destination_history,missing_route_history
0,6879484,16771.0,16831.0,16833.0,19155.0


In [54]:
con.sql("""
SELECT
    MIN(prior_route_flights) AS min_history,
    PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY prior_route_flights) AS p10,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY prior_route_flights) AS p25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY prior_route_flights) AS median,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY prior_route_flights) AS p75,
    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY prior_route_flights) AS p90,
    MAX(prior_route_flights) AS max_history
FROM modeling_with_history
WHERE prior_route_flights IS NOT NULL
""").df()

,min_history,p10,p25,median,p75,p90,max_history
0,1.0,116.0,333.0,862.0,1860.0,3317.0,11187.0


In [55]:
system_daily = con.sql("""
SELECT
    FlightDate,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY FlightDate
""")

In [56]:
system_history = con.sql("""
SELECT
    FlightDate,

    SUM(daily_flights) OVER (
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_system_flights,

    SUM(daily_delayed_flights) OVER (
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_system_delays

FROM system_daily
""")

In [57]:
system_history_rates = con.sql("""
SELECT
    FlightDate,
    prior_system_flights,
    prior_system_delays,

    CASE
        WHEN prior_system_flights > 0
        THEN prior_system_delays * 1.0 / prior_system_flights
        ELSE NULL
    END AS system_historical_delay_rate

FROM system_history
""")

In [58]:
con.sql("""
SELECT *
FROM system_history_rates
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,prior_system_flights,prior_system_delays,system_historical_delay_rate
0,2025-01-01,NaN,NaN,NaN
1,2025-01-02,16771.0,2566.0,0.153002
2,2025-01-03,36202.0,5851.0,0.161621
3,2025-01-04,55430.0,11016.0,0.198737
4,2025-01-05,73687.0,16474.0,0.223567
5,2025-01-06,91838.0,22488.0,0.244866
6,2025-01-07,108654.0,29240.0,0.269111
7,2025-01-08,124099.0,33117.0,0.266860
8,2025-01-09,140170.0,35490.0,0.253193
9,2025-01-10,156107.0,39323.0,0.251898


In [59]:
system_daily = con.sql("""
SELECT
    FlightDate,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY FlightDate
""")

In [60]:
system_history = con.sql("""
SELECT
    FlightDate,

    SUM(daily_flights) OVER (
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_system_flights,

    SUM(daily_delayed_flights) OVER (
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_system_delays

FROM system_daily
""")

In [61]:
system_history_rates = con.sql("""
SELECT
    FlightDate,
    prior_system_flights,
    prior_system_delays,

    CASE
        WHEN prior_system_flights > 0
        THEN prior_system_delays * 1.0 / prior_system_flights
        ELSE NULL
    END AS system_historical_delay_rate

FROM system_history
""")

In [62]:
con.sql("""
SELECT *
FROM system_history_rates
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,prior_system_flights,prior_system_delays,system_historical_delay_rate
0,2025-01-01,NaN,NaN,NaN
1,2025-01-02,16771.0,2566.0,0.153002
2,2025-01-03,36202.0,5851.0,0.161621
3,2025-01-04,55430.0,11016.0,0.198737
4,2025-01-05,73687.0,16474.0,0.223567
5,2025-01-06,91838.0,22488.0,0.244866
6,2025-01-07,108654.0,29240.0,0.269111
7,2025-01-08,124099.0,33117.0,0.266860
8,2025-01-09,140170.0,35490.0,0.253193
9,2025-01-10,156107.0,39323.0,0.251898


In [63]:
modeling_with_all_history = con.sql("""
SELECT
    m.*,
    s.prior_system_flights,
    s.system_historical_delay_rate

FROM modeling_with_history m

LEFT JOIN system_history_rates s
    ON m.FlightDate = s.FlightDate
""")

In [64]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM modeling_with_history) AS before_join,
    (SELECT COUNT(*) FROM modeling_with_all_history) AS after_join
""").df()

,before_join,after_join
0,6879484,6879484


In [65]:
modeling_final_features = con.sql("""
SELECT
    *,

    COALESCE(
        carrier_historical_delay_rate,
        system_historical_delay_rate
    ) AS carrier_delay_rate_final,

    COALESCE(
        origin_historical_delay_rate,
        system_historical_delay_rate
    ) AS origin_delay_rate_final,

    COALESCE(
        destination_historical_delay_rate,
        system_historical_delay_rate
    ) AS destination_delay_rate_final,

    COALESCE(
        route_historical_delay_rate,
        origin_historical_delay_rate,
        destination_historical_delay_rate,
        carrier_historical_delay_rate,
        system_historical_delay_rate
    ) AS route_delay_rate_final,

    CASE WHEN carrier_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS carrier_history_missing,

    CASE WHEN origin_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS origin_history_missing,

    CASE WHEN destination_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS destination_history_missing,

    CASE WHEN route_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS route_history_missing

FROM modeling_with_all_history
""")

In [66]:
final_history_missingness = con.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN carrier_delay_rate_final IS NULL THEN 1 ELSE 0 END)
        AS missing_carrier_final,

    SUM(CASE WHEN origin_delay_rate_final IS NULL THEN 1 ELSE 0 END)
        AS missing_origin_final,

    SUM(CASE WHEN destination_delay_rate_final IS NULL THEN 1 ELSE 0 END)
        AS missing_destination_final,

    SUM(CASE WHEN route_delay_rate_final IS NULL THEN 1 ELSE 0 END)
        AS missing_route_final

FROM modeling_final_features
""").df()

final_history_missingness

,total_rows,missing_carrier_final,missing_origin_final,missing_destination_final,missing_route_final
0,6879484,16771.0,16771.0,16771.0,16771.0


In [71]:
modeling_ready = con.sql("""
SELECT *
FROM modeling_final_features
WHERE system_historical_delay_rate IS NOT NULL
""")

In [72]:
con.sql("""
SELECT
    COUNT(*) AS modeling_rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    COUNT(DISTINCT FlightDate) AS days
FROM modeling_ready
""").df()

,modeling_rows,first_date,last_date,days
0,6862713,2025-01-02,2025-12-31,364


In [73]:
baseline_features = [
    # Schedule / calendar
    "Month",
    "DayOfWeek",
    "scheduled_dep_hour",
    "scheduled_arr_hour",
    "is_weekend",

    # Flight characteristics
    "Reporting_Airline",
    "Origin",
    "Dest",
    "route",
    "CRSElapsedTime",
    "Distance",

    # Leakage-safe historical performance
    "carrier_delay_rate_final",
    "origin_delay_rate_final",
    "destination_delay_rate_final",
    "route_delay_rate_final",

    # Amount of historical evidence
    "prior_carrier_flights",
    "prior_origin_flights",
    "prior_destination_flights",
    "prior_route_flights",

    # Cold-start indicators
    "carrier_history_missing",
    "origin_history_missing",
    "destination_history_missing",
    "route_history_missing"
]

target = "significant_arrival_delay"

In [74]:
leakage_columns = {
    "DepTime",
    "DepDelay",
    "DepDelayMinutes",
    "DepDel15",
    "DepartureDelayGroups",
    "TaxiOut",
    "WheelsOff",
    "WheelsOn",
    "TaxiIn",
    "ArrTime",
    "ArrDelay",
    "ArrDelayMinutes",
    "ArrDel15",
    "ArrivalDelayGroups",
    "ActualElapsedTime",
    "AirTime",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
}

leaked_features = set(baseline_features) & leakage_columns

print("Leakage columns found:", leaked_features)

Leakage columns found: set()


In [75]:
available_columns = set(
    modeling_ready.columns
)

missing_features = [
    col
    for col in baseline_features
    if col not in available_columns
]

print("Missing features:", missing_features)

Missing features: []


In [76]:
train_final = con.sql("""
SELECT *
FROM modeling_ready
WHERE Month BETWEEN 1 AND 9
""")

validation_final = con.sql("""
SELECT *
FROM modeling_ready
WHERE Month BETWEEN 10 AND 11
""")

test_final = con.sql("""
SELECT *
FROM modeling_ready
WHERE Month = 12
""")

In [77]:
final_split_summary = con.sql("""
SELECT
    'train' AS split,
    COUNT(*) AS rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    ROUND(AVG(significant_arrival_delay) * 100, 2) AS delay_rate_pct
FROM train_final

UNION ALL

SELECT
    'validation',
    COUNT(*),
    MIN(FlightDate),
    MAX(FlightDate),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM validation_final

UNION ALL

SELECT
    'test',
    COUNT(*),
    MIN(FlightDate),
    MAX(FlightDate),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM test_final
""").df()

final_split_summary

,split,rows,first_date,last_date,delay_rate_pct
0,train,5133923,2025-01-02,2025-09-30,22.25
1,validation,1156866,2025-10-01,2025-11-30,20.44
2,test,571924,2025-12-01,2025-12-31,26.77


In [78]:
categorical_features = [
    "Reporting_Airline",
    "Origin",
    "Dest"
]

numeric_features = [
    "Month",
    "DayOfWeek",
    "scheduled_dep_hour",
    "scheduled_arr_hour",
    "is_weekend",
    "CRSElapsedTime",
    "Distance",

    "carrier_delay_rate_final",
    "origin_delay_rate_final",
    "destination_delay_rate_final",
    "route_delay_rate_final",

    "prior_carrier_flights",
    "prior_origin_flights",
    "prior_destination_flights",
    "prior_route_flights",

    "carrier_history_missing",
    "origin_history_missing",
    "destination_history_missing",
    "route_history_missing"
]

baseline_v1_features = (
    categorical_features
    + numeric_features
)

len(baseline_v1_features)

22

In [79]:
null_check = con.sql("""
SELECT
    SUM(CASE WHEN CRSElapsedTime IS NULL THEN 1 ELSE 0 END)
        AS missing_crs_elapsed,

    SUM(CASE WHEN Distance IS NULL THEN 1 ELSE 0 END)
        AS missing_distance,

    SUM(CASE WHEN scheduled_dep_hour IS NULL THEN 1 ELSE 0 END)
        AS missing_dep_hour,

    SUM(CASE WHEN scheduled_arr_hour IS NULL THEN 1 ELSE 0 END)
        AS missing_arr_hour,

    SUM(CASE WHEN prior_carrier_flights IS NULL THEN 1 ELSE 0 END)
        AS missing_prior_carrier,

    SUM(CASE WHEN prior_origin_flights IS NULL THEN 1 ELSE 0 END)
        AS missing_prior_origin,

    SUM(CASE WHEN prior_destination_flights IS NULL THEN 1 ELSE 0 END)
        AS missing_prior_destination,

    SUM(CASE WHEN prior_route_flights IS NULL THEN 1 ELSE 0 END)
        AS missing_prior_route

FROM modeling_ready
""").df()

null_check

,missing_crs_elapsed,missing_distance,missing_dep_hour,missing_arr_hour,missing_prior_carrier,missing_prior_origin,missing_prior_destination,missing_prior_route
0,0.0,0.0,0.0,0.0,0.0,60.0,62.0,2384.0


In [80]:
ml_ready = con.sql("""
SELECT
    *,
    
    COALESCE(prior_carrier_flights, 0) AS prior_carrier_flights_ml,
    COALESCE(prior_origin_flights, 0) AS prior_origin_flights_ml,
    COALESCE(prior_destination_flights, 0) AS prior_destination_flights_ml,
    COALESCE(prior_route_flights, 0) AS prior_route_flights_ml

FROM modeling_ready
""")

In [81]:
numeric_features = [
    "Month",
    "DayOfWeek",
    "scheduled_dep_hour",
    "scheduled_arr_hour",
    "is_weekend",
    "CRSElapsedTime",
    "Distance",

    "carrier_delay_rate_final",
    "origin_delay_rate_final",
    "destination_delay_rate_final",
    "route_delay_rate_final",

    "prior_carrier_flights_ml",
    "prior_origin_flights_ml",
    "prior_destination_flights_ml",
    "prior_route_flights_ml",

    "carrier_history_missing",
    "origin_history_missing",
    "destination_history_missing",
    "route_history_missing"
]

baseline_v1_features = categorical_features + numeric_features

print("Baseline features:", len(baseline_v1_features))

Baseline features: 22


In [82]:
null_counts = con.sql("""
SELECT
    COUNT(*) AS rows_with_any_null
FROM ml_ready
WHERE
    Reporting_Airline IS NULL
    OR Origin IS NULL
    OR Dest IS NULL
    OR Month IS NULL
    OR DayOfWeek IS NULL
    OR scheduled_dep_hour IS NULL
    OR scheduled_arr_hour IS NULL
    OR is_weekend IS NULL
    OR CRSElapsedTime IS NULL
    OR Distance IS NULL
    OR carrier_delay_rate_final IS NULL
    OR origin_delay_rate_final IS NULL
    OR destination_delay_rate_final IS NULL
    OR route_delay_rate_final IS NULL
    OR prior_carrier_flights_ml IS NULL
    OR prior_origin_flights_ml IS NULL
    OR prior_destination_flights_ml IS NULL
    OR prior_route_flights_ml IS NULL
    OR carrier_history_missing IS NULL
    OR origin_history_missing IS NULL
    OR destination_history_missing IS NULL
    OR route_history_missing IS NULL
    OR significant_arrival_delay IS NULL
""").df()

null_counts

,rows_with_any_null
0,0
